# RLS Role Management — Power BI Semantic Model

This notebook automates the creation, replacement, and member assignment of Row Level Security (RLS) roles in a Power BI Semantic Model using `semantic-link-labs` and the Tabular Object Model (TOM).

---

## Overview

| Step | Description | Function |
|------|-------------|----------|
| 1 | Install dependencies | — |
| 2 | Configure dataset, workspace, roles | `config`, `global_filters` |
| 3 | Load RLS source table | `rls` DataFrame |
| 4 | Create or replace roles + DAX filters | `create_or_replace_roles()` |
| 5 | Add members to roles | `add_members_to_roles()` |

---

## Design decisions

- **Filter on Dimension Tables (DT), not Fact Tables (FT)** — filters propagate through relationships automatically, avoiding full fact table scans on every query.
- **Static roles over dynamic RLS** — static DAX expressions are evaluated once at connection time, not on every query per user.
- **Global filters** are always applied to every role on their own fixed table, independent of the role-specific filter.
- **Members are saved one at a time** — isolates invalid UPNs without blocking valid ones.
- **Failed members** are exported as a JSON report to the Lakehouse `Files` folder.

## 1 — Install dependencies

In [ ]:
%pip install semantic-link-labs -q

## 2 — Imports

In [ ]:
import json
from pyspark.sql import functions as F
from sempy_labs.tom import connect_semantic_model

## 3 — Configuration

### Parameters

| Parameter | Description |
|-----------|-------------|
| `dataset` | Name or ID of the target Power BI Semantic Model |
| `workspace` | Name or ID of the Fabric workspace |
| `global_filters` | Filters applied to **every** role, each on their own fixed table |
| `config` | Role definitions — one entry per dimension to secure |

### `config` structure

```python
config = {
    "<column_in_rls_df>": {
        "table":  "<DT table name in semantic model>",
        "prefix": "<role name prefix>",
        "column": "<column name in semantic model table>"
    }
}
```

### `global_filters` structure

```python
global_filters = [
    {
        "table":  "<table name in semantic model>",
        "column": "<column name>",
        "value":  "<filter value>"
    }
]
```

In [ ]:
# ── Target semantic model ─────────────────────────────────────────────────────
dataset   = ""  # Name or ID of the semantic model
workspace = ""  # Name or ID of the Fabric workspace

# ── Global filters — applied to every role on their own fixed table ───────────
# These are never merged with role-specific filters even if the table matches.
# Multiple global filters are supported.
global_filters = [
    {
        "table":  "DT_Customer",   # Dimension table where the global filter column lives
        "column": "Is_Consolidated",
        "value":  "Consolidated"
    },
    # Add more global filters as needed, e.g.:
    # {
    #     "table":  "DT_Product",
    #     "column": "Is_Active",
    #     "value":  "True"
    # },
]

# ── Role configuration ────────────────────────────────────────────────────────
# Key    = column name in the RLS DataFrame (rls)
# table  = Dimension Table (DT_*) in the semantic model — always filter on DT,
#          never on Fact Tables (FT_*); relationships propagate the filter
# prefix = role name prefix; roles will be named <prefix>_<value>
# column = column in the semantic model DT to filter on
config = {
    "CountryCode": {
        "table":  "DT_Customer",
        "prefix": "RLS_CountryCode",
        "column": "CountryCode"
    },
    "TerritoryCode": {
        "table":  "DT_Customer",
        "prefix": "RLS_TerritoryCode",
        "column": "TerritoryCode"
    },
    "BrandingCode": {
        "table":  "DT_Product",
        "prefix": "RLS_BrandingCode",
        "column": "BrandingCode"
    },
    # Add more dimensions as needed, e.g.:
    # "Zone": {
    #     "table":  "DT_Customer",
    #     "prefix": "RLS_Zone",
    #     "column": "Zone"
    # },
    # "SalesChannel": {
    #     "table":  "DT_SalesChannel",
    #     "prefix": "RLS_SalesChannel",
    #     "column": "SalesChannel"
    # },
}

## 4 — Load RLS source table

The RLS DataFrame (`rls`) must contain:

| Column | Description |
|--------|-------------|
| `Username` | UPN email of the user (`user@domain.com`) |
| `<config_key>` | One column per entry in `config` (e.g. `CountryCode`, `TerritoryCode`) |

> Replace the cell below with your actual Lakehouse table read.

In [ ]:
# ── Load RLS source DataFrame ─────────────────────────────────────────────────
# Replace with your actual table path or query
rls = spark.table("your_lakehouse.your_rls_table")

# Preview
display(rls.limit(10))

## 5 — Helper functions

In [ ]:
def build_dax_filters(settings, value, global_filters):
    """
    Builds a list of (table, dax_expression) tuples for a given role.

    Rules:
    - Role-specific filter and global filters that share the same table
      are combined with && into a single set_rls call.
    - Filters on different tables always produce separate set_rls calls.
    - set_rls replaces any existing filter on a table, so same-table
      filters must be merged to avoid overwriting each other.

    Parameters
    ----------
    settings : dict
        Config entry for the current dimension.
    value : str
        The dimension value for this role (e.g. "AU", "MEAI").
    global_filters : list[dict]
        List of global filter definitions.

    Returns
    -------
    list of (table, dax_expression) tuples
    """
    filters_by_table = {}

    # Role-specific filter on its own table
    role_column = settings.get("column")
    if role_column:
        table = settings["table"]
        filters_by_table.setdefault(table, []).append(
            f'[{role_column}] = "{value}"'
        )

    # Global filters — merged into same table if it matches, else separate
    for gf in global_filters:
        table = gf["table"]
        expr  = f'[{gf["column"]}] = "{gf["value"]}"'
        filters_by_table.setdefault(table, []).append(expr)

    # Combine multiple filters on the same table with &&
    return [
        (table, " && ".join(f"({expr})" for expr in exprs))
        for table, exprs in filters_by_table.items()
    ]


def create_or_replace_roles(config, dataset, workspace, global_filters, rls, config_keys=None):
    """
    Creates or replaces RLS roles in the semantic model.

    - Idempotent: existing roles with the same name are removed and recreated.
    - One SaveChanges() call at the end for efficiency.
    - Does NOT add members — run add_members_to_roles() separately.

    Parameters
    ----------
    config : dict
        Role configuration.
    dataset : str
        Semantic model name or ID.
    workspace : str
        Fabric workspace name or ID.
    global_filters : list[dict]
        Global filters applied to every role.
    rls : pyspark.sql.DataFrame
        RLS source DataFrame.
    config_keys : list[str], optional
        Subset of config keys to process. If None, all keys are processed.
    """
    selected = {
        k: v for k, v in config.items()
        if config_keys is None or k in config_keys
    }

    if config_keys:
        print(f"🎯 Processing {len(selected)}/{len(config)} config(s): {list(selected.keys())}\n")
    else:
        print(f"🎯 Processing all {len(selected)} config(s)\n")

    with connect_semantic_model(dataset=dataset, readonly=False, workspace=workspace) as tom:

        for config_key, settings in selected.items():
            prefix = settings["prefix"]

            unique_values = (
                rls.select(config_key)
                .where(F.col(config_key).isNotNull())
                .distinct()
                .collect()
            )

            print(f"\n📂 {config_key} — {len(unique_values)} distinct value(s)")

            for row in unique_values:
                value     = str(row[config_key])
                role_name = f"{prefix}_{value.replace(' ', '_')}"
                filters   = build_dax_filters(settings, value, global_filters)

                existing_role = tom.model.Roles.Find(role_name)
                if existing_role is not None:
                    print(f"  🔄 Replacing: {role_name}")
                    tom.model.Roles.Remove(existing_role)
                else:
                    print(f"  ✅ Creating:  {role_name}")

                tom.add_role(role_name=role_name)

                for table, dax_filter in filters:
                    tom.set_rls(
                        role_name=role_name,
                        table_name=table,
                        filter_expression=dax_filter
                    )
                    print(f"     🔒 {table}: {dax_filter}")

        tom.model.SaveChanges()
        print("\n✅ All roles saved successfully.")


def add_members_to_roles(config, dataset, workspace, rls, username_col="Username", config_keys=None):
    """
    Adds members to existing RLS roles from the RLS source DataFrame.

    - Pre-validates all UPNs before touching the model.
    - Opens a fresh TOM connection per member to avoid stale state
      from a failed SaveChanges() poisoning subsequent saves.
    - Invalid UPNs are skipped and collected in the failure report.
    - Failed members are exported as JSON to Lakehouse Files.

    Parameters
    ----------
    config : dict
        Role configuration.
    dataset : str
        Semantic model name or ID.
    workspace : str
        Fabric workspace name or ID.
    rls : pyspark.sql.DataFrame
        RLS source DataFrame containing Username and dimension columns.
    username_col : str, default 'Username'
        Column name in rls containing UPN emails.
    config_keys : list[str], optional
        Subset of config keys to process. If None, all keys are processed.

    Returns
    -------
    list[dict]
        List of failed members with role, username, and reason.
    """
    selected = {
        k: v for k, v in config.items()
        if config_keys is None or k in config_keys
    }

    if config_keys:
        print(f"🎯 Processing {len(selected)}/{len(config)} config(s): {list(selected.keys())}\n")
    else:
        print(f"🎯 Processing all {len(selected)} config(s)\n")

    # ── Pre-validate all UPNs before touching the model ───────────────────────
    all_upns = (
        rls.select(username_col)
        .where(F.col(username_col).isNotNull())
        .distinct()
        .collect()
    )

    valid_upns   = set()
    invalid_upns = set()

    for row in all_upns:
        upn = str(row[username_col]).strip()
        if "@" in upn and " " not in upn:
            valid_upns.add(upn)
        else:
            invalid_upns.add(upn)

    print(f"📊 UPN validation: {len(valid_upns)} valid, {len(invalid_upns)} invalid\n")

    if invalid_upns:
        print("  ⚠️  Invalid UPNs (will be skipped):")
        for u in sorted(invalid_upns):
            print(f"     - {u}")
        print()

    failed_members = []

    for config_key, settings in selected.items():
        prefix = settings["prefix"]

        unique_values = (
            rls.select(config_key)
            .where(F.col(config_key).isNotNull())
            .distinct()
            .collect()
        )

        print(f"\n📂 {config_key} — {len(unique_values)} distinct value(s)")

        for row in unique_values:
            value     = str(row[config_key])
            role_name = f"{prefix}_{value.replace(' ', '_')}"

            raw_members = (
                rls.where(F.col(config_key) == value)
                .select(username_col)
                .where(F.col(username_col).isNotNull())
                .distinct()
                .collect()
            )

            for r in raw_members:
                upn = str(r[username_col]).strip()

                # Skip pre-validation failures
                if upn not in valid_upns:
                    print(f"  ⚠️  Skipped (invalid format): {upn}")
                    failed_members.append({
                        "role":     role_name,
                        "username": upn,
                        "reason":   "Invalid UPN format"
                    })
                    continue

                # Fresh TOM connection per member — prevents stale failed
                # state from a previous SaveChanges() failure poisoning
                # subsequent valid members in the same session.
                try:
                    with connect_semantic_model(
                        dataset=dataset, readonly=False, workspace=workspace
                    ) as tom:
                        if tom.model.Roles.Find(role_name) is None:
                            print(f"  ⚠️  Role not found, skipping: {role_name}")
                            break

                        tom.add_role_member(
                            role_name=role_name,
                            member=upn,
                            role_member_type="User"
                        )
                        tom.model.SaveChanges()
                        print(f"  ✅ Saved: {upn} → {role_name}")

                except Exception as e:
                    error_msg = str(e)[:200]
                    print(f"  ❌ Failed: {upn} → {role_name} | {error_msg}")
                    failed_members.append({
                        "role":     role_name,
                        "username": upn,
                        "reason":   error_msg
                    })

    # ── Final report ──────────────────────────────────────────────────────────
    print("\n" + "─" * 60)
    print("✅ Done.")

    if failed_members:
        print(f"\n❌ {len(failed_members)} member(s) failed — exporting report...\n")
        print(json.dumps(failed_members, indent=2))

        failed_df = spark.createDataFrame(failed_members)
        (
            failed_df
            .coalesce(1)
            .write
            .mode("overwrite")
            .json("Files/rls_failed_members")
        )
        print("\n💾 Report saved to: Files/rls_failed_members/")
    else:
        print("🎉 All members added successfully — no failures!")

    return failed_members

## 6 — Debug (optional)

Run this cell to preview the DAX filters that will be generated before touching the model.

In [ ]:
# ── Preview DAX filters per config key ───────────────────────────────────────
for config_key, settings in config.items():
    print(f"\n🔵 Config key: {config_key}")
    print(f"   Role table:  {settings['table']}")

    sample = (
        rls.select(config_key)
        .where(F.col(config_key).isNotNull())
        .first()
    )

    if sample:
        value   = str(sample[config_key])
        filters = build_dax_filters(settings, value, global_filters)
        print(f"   Sample value: {value}")
        for table, dax in filters:
            print(f"   🔒 {table}: {dax}")

## 7 — Run

### Options

| Parameter | Example | Description |
|-----------|---------|-------------|
| `config_keys=None` | _(default)_ | Process all config entries |
| `config_keys=["CountryCode"]` | Run one config | Process only the specified keys |
| `username_col="Username"` | _(default)_ | Column in `rls` containing UPN emails |

> **Step 1** (roles) is always safe to re-run — it creates or replaces.  
> **Step 2** (members) is optional — comment it out if you only want to update role definitions.

In [ ]:
# ── Step 1: Create or replace roles ──────────────────────────────────────────
create_or_replace_roles(
    config        = config,
    dataset       = dataset,
    workspace     = workspace,
    global_filters= global_filters,
    rls           = rls,
    config_keys   = None,   # None = all; or e.g. ["CountryCode", "BrandingCode"]
)

In [ ]:
# ── Step 2: Add members (optional) ───────────────────────────────────────────
failed = add_members_to_roles(
    config       = config,
    dataset      = dataset,
    workspace    = workspace,
    rls          = rls,
    username_col = "Username",  # Column in rls containing UPN emails
    config_keys  = None,        # None = all; or e.g. ["CountryCode"]
)

## 8 — Review failures (optional)

If any members failed, display them as a DataFrame for easier review.

In [ ]:
# ── Display failed members as a DataFrame ─────────────────────────────────────
if failed:
    display(spark.createDataFrame(failed))
else:
    print("🎉 No failures to review.")